In [1]:
# Cell 1: Imports, Environment Setup, and Directory Management
import sys
import os
import glob
import math
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import pypowsybl as pp
import pypowsybl.network as pn
from OMPython import OMCSessionZMQ

NOTEBOOK_NAME = "OM_test_3"
print(f"[INFO] Detected Notebook Name: {NOTEBOOK_NAME}")

# --- 2. OUTPUT DIRECTORY SETUP ---
current_dir = os.getcwd()
output_dir_name = f"{NOTEBOOK_NAME}_outputs"
OUTPUT_DIR = os.path.join(current_dir, output_dir_name)

# Create folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Output Directory set to: {OUTPUT_DIR}")

# --- 3. PATH CONFIGURATION (INPUTS) ---
# CRITICAL: Use Absolute Paths for inputs so OpenModelica can find them
# even if we change the working directory later.

# Dynawo Path (Update if necessary)
DYNAWO_PKG_PATH = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"

# Model Paths (Assuming .mo files are in the same folder as the notebook)
static_model_path = os.path.join(current_dir, "MyBESS_static.mo").replace("\\", "/")
dynamic_model_path = os.path.join(current_dir, "MyBESS.mo").replace("\\", "/")

# --- 4. INITIALIZE OPENMODELICA ---
try:
    omc = OMCSessionZMQ()
    print("[INFO] OpenModelica Session Started.")

    # A. Load Dynawo
    if os.path.exists(DYNAWO_PKG_PATH):
        print(f"[INFO] Loading Dynawo from: {DYNAWO_PKG_PATH}")
        loaded = omc.sendExpression(f'loadFile("{DYNAWO_PKG_PATH}")')
        if not loaded:
            print("[WARNING] OMPython returned False when loading Dynawo.")
    else:
        print(f"[ERROR] File not found at: {DYNAWO_PKG_PATH}")
        omc.sendExpression("loadModel(Dynawo)")

    # B. Load User Models
    omc.sendExpression(f'loadFile("{static_model_path}")')
    omc.sendExpression(f'loadFile("{dynamic_model_path}")')
    print("[SUCCESS] BESS Models loaded successfully.")

    # C. Change OpenModelica Working Directory
    # This forces all simulation results (.mat, .log) to go into the output folder
    # We use forward slashes for compatibility
    safe_output_dir = OUTPUT_DIR.replace("\\", "/")
    omc.sendExpression(f'cd("{safe_output_dir}")')
    print(f"[INFO] OpenModelica working directory changed to outputs folder.")

except Exception as e:
    print(f"[ERROR] Error connecting to OpenModelica: {e}")

print("Environment ready.")

[INFO] Detected Notebook Name: OM_test_3
[INFO] Output Directory set to: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_3_outputs
[INFO] OpenModelica Session Started.
[INFO] Loading Dynawo from: /home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo
[SUCCESS] BESS Models loaded successfully.
[INFO] OpenModelica working directory changed to outputs folder.
Environment ready.


In [2]:
# Cell 2: Network Logic Helper
# Encapsulates PyPowSyBl logic to create the network and solve the power flow


def solve_static_network(p_setpoint_pu, s_base=100.0, nom_v=110.0):
    """
    Creates the BESS network topology, sets the active power,
    runs the AC Loadflow, and returns initialization values.
    """
    # 1. Calculate Base Impedance
    z_base = (nom_v**2) / s_base

    # 2. Create Empty Network
    network = pp.network.create_empty()

    # Substations & Voltage Levels
    network.create_substations(id="Sub_Grid", name="Grid", country="FR", tso="TSO")
    network.create_substations(id="Sub_BESS", name="BESS", country="FR", tso="TSO")
    network.create_voltage_levels(
        id="Grid_VL", substation_id="Sub_Grid", topology_kind="BUS_BREAKER", nominal_v=nom_v
    )
    network.create_voltage_levels(
        id="BESS_VL", substation_id="Sub_BESS", topology_kind="BUS_BREAKER", nominal_v=nom_v
    )

    # Buses & Lines
    network.create_buses(id="Bus_Infinite", voltage_level_id="Grid_VL")
    network.create_buses(id="Bus_BESS", voltage_level_id="BESS_VL")

    # Line parameters derived from MyBESS_static.mo
    network.create_lines(
        id="Line_Connect",
        voltage_level1_id="Grid_VL",
        bus1_id="Bus_Infinite",
        voltage_level2_id="BESS_VL",
        bus2_id="Bus_BESS",
        x=0.0000020661 * z_base,
        r=0.0,
    )

    # Generators (Infinite Bus & BESS)
    network.create_generators(
        id="InfiniteBus",
        voltage_level_id="Grid_VL",
        bus_id="Bus_Infinite",
        target_v=1.0 * nom_v,
        target_p=0.0,
        voltage_regulator_on=True,
        min_p=-9999.0,
        max_p=9999.0,
    )

    # Convert PU setpoint to MW
    p_target_mw = p_setpoint_pu * s_base

    network.create_generators(
        id="GenPV",
        voltage_level_id="BESS_VL",
        bus_id="Bus_BESS",
        target_p=p_target_mw,
        target_v=1.0 * nom_v,
        voltage_regulator_on=True,
        min_p=-100.0,
        max_p=100.0,
        rated_s=s_base,
    )

    # 3. Run AC Loadflow
    params = pp.loadflow.Parameters(
        distributed_slack=False,
        provider_parameters={"slackBusSelectionMode": "NAME", "slackBusesIds": "InfiniteBus"},
    )
    results = pp.loadflow.run_ac(network, parameters=params)

    # Check Convergence (Robust check)
    status_name = (
        results[0].status.name if hasattr(results[0].status, "name") else str(results[0].status)
    )
    if status_name != "CONVERGED":
        raise RuntimeError(f"Power Flow failed to converge. Status: {status_name}")

    # 4. Extract Initialization Values
    gens = network.get_generators(all_attributes=True)
    buses = network.get_buses(all_attributes=True)

    # Get physical values
    p_val = gens.at["GenPV", "p"]
    q_val = gens.at["GenPV", "q"]
    bus_id = gens.at["GenPV", "bus_id"]
    v_val = buses.at[bus_id, "v_mag"]
    angle_deg = buses.at[bus_id, "v_angle"]

    # Return dictionary in PU and Radians
    return {
        "P0_pu": p_val / s_base,
        "Q0_pu": q_val / s_base,
        "V0_pu": v_val / nom_v,
        "Angle_rad": math.radians(angle_deg),
    }

In [3]:
# Cell 3: Simulation Logic Helper
# Sends parameters to OpenModelica, runs simulation, and reads results safely


def read_om_mat_file(filepath, vars_to_read):
    """
    Reads an OpenModelica .mat result file.
    Specifically fixed to handle 1D and 2D name matrices.
    """
    try:
        # Load MAT file
        mat = scipy.io.loadmat(filepath, squeeze_me=False)

        # --- 1. GET RAW MATRICES ---
        raw_names = mat.get("name")
        raw_data = mat.get("data_2")
        data_info = mat.get("dataInfo")

        if raw_names is None or raw_data is None or data_info is None:
            print("[ERROR] Invalid MAT file structure.")
            return None

        # --- 2. FIX MATRIX ORIENTATION & DIMENSIONS ---

        # Check if raw_names is 1D or 2D safely
        if raw_names.ndim == 1:
            # If 1D, we treat it as a single variable or a flat list
            num_vars = raw_names.shape[0]
            names_is_1d = True
        else:
            # If 2D, standard logic: check if it needs transposing
            if raw_names.shape[0] < raw_names.shape[1] and raw_names.shape[0] < 100:
                raw_names = raw_names.T
            num_vars = raw_names.shape[0]
            names_is_1d = False

        # Fix dataInfo orientation (Must be N_vars x 4)
        if data_info.shape[0] == 4 and data_info.shape[1] == num_vars:
            data_info = data_info.T

        # Fix raw_data orientation (Must be N_rows x TimeSteps)
        # We find the max index referenced in dataInfo
        max_idx_needed = np.max(np.abs(data_info[:, 1])) if data_info.size > 0 else 0
        if raw_data.shape[0] < max_idx_needed:
            raw_data = raw_data.T

        # --- 3. BUILD VARIABLE MAP ---
        var_map = {}

        def clean_name(row_data):
            try:
                # If row_data is a single element (1D array case)
                if np.isscalar(row_data):
                    return str(row_data).strip()
                # If it's a row of characters
                s = "".join([chr(int(c)) if c > 0 else "" for c in row_data.flatten()])
                return s.strip()
            except:
                return str(row_data).strip()

        for i in range(min(num_vars, data_info.shape[0])):
            if names_is_1d:
                name = clean_name(raw_names[i])
            else:
                name = clean_name(raw_names[i, :])

            if not name:
                continue

            # Column 1 of dataInfo is the row index in data_2
            data_idx = int(data_info[i, 1])

            if data_idx == 0:
                var_map[name] = (0, 1)
            else:
                sign = 1
                if data_idx < 0:
                    sign = -1
                    data_idx = abs(data_idx)
                var_map[name] = (data_idx - 1, sign)

        # --- 4. EXTRACT DATA ---
        extracted_data = []

        # Time is usually row 0 or named "time"
        if "time" in var_map:
            idx, _ = var_map["time"]
            t_row = raw_data[idx]
        else:
            t_row = raw_data[0]

        extracted_data.append(t_row)

        for var in vars_to_read:
            if var == "time":
                continue

            if var in var_map:
                idx, sign = var_map[var]
                if idx < raw_data.shape[0]:
                    extracted_data.append(raw_data[idx] * sign)
                else:
                    extracted_data.append(np.zeros_like(t_row))
            else:
                extracted_data.append(np.zeros_like(t_row))

        return extracted_data

    except Exception as e:
        print(f"[ERROR] Failed to read MAT: {e}")
        return None


def run_simulation(init_values, model_name="Dynawo.Examples.BESS.WECC.MyBESS_static"):
    print(f"[INFO] Preparing simulation for {model_name}...")

    # Update parameters
    cmds = [
        f"setParameterValue({model_name}, GenPV.PGen0Pu, {init_values['P0_pu']})",
        f"setParameterValue({model_name}, GenPV.U0Pu, {init_values['V0_pu']})",
    ]
    for cmd in cmds:
        omc.sendExpression(cmd)

    # Simulate
    print("   [INFO] Running simulation (5s)...")
    result = omc.sendExpression(f"simulate({model_name}, stopTime=5.0, numberOfIntervals=500)")

    result_file = ""
    if isinstance(result, dict) and "resultFile" in result:
        result_file = result["resultFile"]

    if not result_file or not os.path.exists(result_file):
        if result_file and not os.path.isabs(result_file):
            result_file = os.path.join(OUTPUT_DIR, result_file)

        if not os.path.exists(result_file):
            print("[ERROR] Simulation Failed. No result file found.")
            return None

    print(f"[SUCCESS] Simulation successful! Parsing file: {result_file}")

    vars_to_read = ["time", "GenPV.PGen", "GenPV.QGen", "GenPV.terminal.v"]
    return read_om_mat_file(result_file, vars_to_read)

In [4]:
# Cell 4: Plotting Helper (Updated)


def plot_simulation_results(data):
    if data is None or len(data) < 4:
        print("[WARNING] No valid data to plot.")
        return

    # Unpack data
    t_axis = data[0]
    p_curve = data[1]
    q_curve = data[2]
    v_curve = data[3]

    # Create Figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Power
    ax1.plot(t_axis, p_curve, label="Active Power (P)", linewidth=2)
    ax1.plot(t_axis, q_curve, label="Reactive Power (Q)", linewidth=2)
    ax1.set_title("Active & Reactive Power Response")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Power (pu)")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.6)

    # Plot 2: Voltage
    ax2.plot(t_axis, v_curve, label="Terminal Voltage", color="green", linewidth=2)
    ax2.set_title("Voltage Magnitude")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Voltage (pu)")

    # Dynamic Y-axis scaling
    if len(v_curve) > 0:
        v_min, v_max = np.min(v_curve), np.max(v_curve)
        margin = max((v_max - v_min) * 0.1, 0.001)
        ax2.set_ylim(v_min - margin, v_max + margin)

    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()

In [5]:
# Cell 5: Interactive Pipeline
# Connects Static Loadflow -> Initialization -> Dynamic Simulation -> Visualization


def run_pipeline(p_setpoint):
    print(f"\n[STEP 1] Static Loadflow (PyPowSyBl)")
    print(f"   Target P = {p_setpoint} pu")

    try:
        init_vals = solve_static_network(p_setpoint)
        print(f"   [SUCCESS] Converged.")
        print(f"   Calculated Initial Q: {init_vals['Q0_pu']:.4f} pu")
        print(f"   Calculated Angle:     {init_vals['Angle_rad']:.4f} rad")
    except Exception as e:
        print(f"   [ERROR] Loadflow Failed: {e}")
        return

    print(f"\n[STEP 2] Dynamic Simulation (OpenModelica)")
    # Note: Ensure we simulate the static wrapper or the model that accepts the params
    # Using 'MyBESS_static' as it contains the GenPV we are parameterizing
    sim_data = run_simulation(init_vals, model_name="Dynawo.Examples.BESS.WECC.MyBESS_static")

    print(f"\n[STEP 3] Visualization")
    plot_simulation_results(sim_data)


# Launch the Widget
print("Adjust the slider to change the BESS Active Power Setpoint:")
interact(
    run_pipeline,
    p_setpoint=widgets.FloatSlider(
        value=0.03,
        min=-0.9,
        max=0.9,
        step=0.05,
        description="P Setpoint (pu)",
        continuous_update=False,  # Wait for mouse release to run
    ),
);

Adjust the slider to change the BESS Active Power Setpoint:


interactive(children=(FloatSlider(value=0.03, continuous_update=False, description='P Setpoint (pu)', max=0.9,…